# Optimization Algorithms Note: 1
#### Abir Hossain | May, 2026
- This notebook has notes on the different optimization algorithms and definition-details about them. The purpose of this notebook is to create a solid introduction to the different methods and usecases along with the reasoning behind.
- Algorithms:
  + First-Order Optimization Methods: The SGD Family
    * SGD
    * SGD with Momentum
    * NAG
    * AdaGrad
    * RMSprop
    * ADAM
    * ADAMW
    * Nadam
    * RAdam
    * LARS/LAMB
    * Lookahead
    * SignSGD
    * LION
    * Schedule Free Optimizers
    * Muon
    * GaLore
    * LOMO

# Optimizer
An optimizer is an algorithm that adjusts the parameters $\theta$ of a model to minimize a loss function $\mathcal{L}(\theta)$. It takes gradients *(and sometimes higher-order information)* computed via backpropagation and converts them into parameter updates.
Every optimizer answers three fundamental questions:  
- Which direction should we move? *(gradient, momentum, curvature)*  
- How far should we step? *(learning rate, adaptive scaling, line search)*  
- What history should we remember? *(none, gradient history, curvature history)*

# First Order Methods
 Use only the gradient $\nabla_\theta \mathcal{L}$.
- ***Computation:*** One forward-backward pass per update
- ***Memory:*** (1–3)$\times$ model parameters (parameters + optional momentum states)
- ***Convergence:*** Linear (O(1/t)  or O(1/$\sqrt{t}$) for convex; no guarantees for non-convex)
- ***Scale:*** Proven at billions of parameters

## Vanilla Stochastic Gradient Descent (SGD)
- **The Update Rule:**
$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta}\mathcal{L}(\theta_t)$  
This coomputes the gradient of the loss with respect to parameters, and step in the opposite direction. The ***stochastic*** part means we estimate the gradient using a mini-batch rather than the full dataset.
- **Pros:**
    + Minimal memory overhead; only stores parameters and gradients
    + Strong theoretical convergence guarantees for convex problems
    + Often achieves better generalization than adaptive methods on vision tasks; with proper tuning
    + The ***gold standard*** baseline
- **Cons:**
    + Slow convergence in narrow valleys, saddle points, and ravines; zigzags inefficiently
    + Highly sensitive to learning rate choice; requires careful scheduling
    + Same learning rate applied to all parameters regardless of their gradient magnitudes or frequencies
    + Struggles with ill-conditioned problems; where curvature varies wildly across dimensions
- **Usecases:**
    + Large-scale vision training (ResNets, ConvNets) where generalization matters
    + When time is available to tune learning rate schedules meticulously 

## SGD with Momentum (Polyak, 1964)
- **The Update Rule**  
$v_{t+1} = \beta v_t+ \nabla_{\theta} \mathcal{L}(\theta_t)$   
$\theta_{t+1} = \theta_t − \eta \cdot v_{t+1}$   
Where:  
    $v_t$  = velocity; accumulated gradient  
    $\beta \in$ [0,1)  = momentum coefficient; typically 0.9  
In a long, narrow valley; gradients oscillate across the short axis but agree along the long axis. Momentum cancels the oscillations and builds speed along the valley. Escapes shallow local minima and plateaus more easily than vanilla SGD.
- **Pros:**
    + Faster convergence in high-curvature directions
    + Reduces variance in stochastic gradients
    + Helps escape saddle points; common in high dimensions
- **Cons:**
    + Adds one hyperparameter; $\beta$
    + Can overshoot minima if momentum is too high
    + May cause divergence with very large learning rates
- **Usecase:**
    The default SGD for deep learning. Almost always preferable to vanilla SGD.

## Nesterov Accelerated Gradient (NAG, 1983)
- **The Update Rule:**  
$v_{t+1} = \beta v_t + \nabla_\theta \mathcal{L}(θ_t − \eta \beta v_t)$     
$\theta_{t+1} = \theta_t - \eta \cdot v_{t+1}$   
Or equivalently in the more common ***lookahead gradient*** formulation:  
$v_{t+1} = \beta v_t + \nabla_\theta \mathcal{L}(\theta_t)$   
$θ_{t+1} = \theta_t − \eta (\beta v_{t+1} + \nabla_\theta \mathcal{L}(\theta_t))$   
NAG works a bit differently than conventional momentum: it first makes a big jump in the direction of the previous velocity, then measures the gradient at that lookahead position and makes a correction. This is like a ball rolling downhill that becomes aware of where it's going to be and slows down before the hill starts sloping up again. It reduces overshooting.
- **Pros:**
    + Provably optimal convergence rate for convex smooth functions: O($\frac{1}{t^2}$)  vs O($\frac{1}{t}$)  for vanilla SGD
    + More stable than classical momentum near minima; resulting in less oscillation
    + Better handling of high curvature
- **Cons:**
    + Slightly more computationally expensive; as it requires gradient at lookahead point
    + The theoretical guarantees don't always transfer to non-convex deep learning landscapes
    + In practice, often similar to well-tuned momentum
- **Usecases:**
    + When using SGD + Momentum, try NAG as a drop-in replacement. *(PyTorch offers nesterov=True in torch.optim.SGD.)*
    + Theoretical settings where convergence guarantees matter.

## AdaGrad (Duchi et al., 2011)
- **The Update Rule:**  
$G_t = G_{t-1} + (\nabla_\theta \mathcal{L}(\theta_t))^2$  
$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t} + \epsilon} \odot \nabla_\theta \mathcal{L}(\theta_t)$  
    + Where $G_t$ is the sum of squared gradients.
    + Parameters with large historical gradients get smaller effective learning rates; as they've been updated a lot already.
    + Parameters with small historical gradients get larger learning rates; as they're rarely updated, so need bigger steps.
    + Ideal for sparse features *(e.g., word embeddings where rare words need larger updates)*.
- **Pros:**
    + Automatically adapts learning rates per parameter
    + Excellent for sparse data and NLP with rare features
- **Cons:**
    + Learning rate decays to zero over time because $G_t$ monotonically accumulates. Training effectively stops prematurely.
    + Rarely used in modern deep learning for this reason.
- **Usecases:**
    + Sparse linear models, traditional NLP feature-based models
    + Mostly superseded by RMSprop and Adam

## RMSprop (Hinton, 2012 — unpublished but widely used)
- **The Update Rule:**  
$E[g^2]_t = \beta E[g^2]_{t-1} + (1 - \beta) (\nabla_\theta \mathcal{L}(\theta_t))^2$   
$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_t} + \epsilon} \odot \nabla_\theta \mathcal{L}(\theta_t)$  
    + Where $E[g^2]$ is the exponentially moving average of squared gradients.
    + Fixes AdaGrad's fatal flaw by replacing the cumulative sum with an exponential moving average. This means recent gradients matter more than ancient ones, and the effective learning rate no longer collapses to zero.
- **Pros:**
    + Works well in non-stationary settings
    + Handles different curvatures across parameters
    + Good default for RNNs before Adam became dominant
- **Cons:**      
    + Still requires global learning rate tuning
    + No momentum on the parameter updates themselves; just on the squared gradients
    + Can get stuck in saddle points
- **Usecases:**
    + RNN training
    + As a component understanding for Adam

## Adam (Adaptive Moment Estimation, Kingma & Ba, 2014)
- **The Update Rule:**  
$m_t = \beta_1 m_{t−1} + (1 − β_1)g_t \; | \;\text{first moment: mean}$   
$v_t = \beta_2 v_{t-1} + (1 - \beta_2)g_t^2 \; | \; \text{second moment: uncentered variance}$   
$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} , \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \; | \; \text{bias correction}$   
$\theta_{t+1} = \theta_t − \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$  
Where:  
    $\beta_1$ = 0.9  (momentum)  
    $\beta_2$ =0.999  (RMSprop-style squared gradient decay)   
    $\epsilon = 10^{−8}$   
Combines momentum (via $m_t$) with adaptive learning rates (via $v_t$). So, ***RMSprop with momentum*** added with ***bias correction for the initial steps*** (since $m_0$ and $v_0$ are initialized to zero, early estimates are biased toward zero).
- **Pros:**
    + Works well right out of the box with default hyperparameters
    + Fast initial progress
    + Handles sparse gradients and noisy problems well
    + The default optimizer for most deep learning practitioners
- **Cons:**
    + Generalizes worse than SGD+Momentum on some vision tasks (ImageNet). Adam often finds sharper minima that don't generalize as well.
    + Weight decay doesn't work correctly with Adam (L2 regularization is not equivalent to weight decay when gradients are normalized)
    + Can diverge in some non-convex settings *("Adam can fail to converge" paper by Reddi et al., 2018)*
    + ***Memory overhead:*** stores two moving averages per parameter
- **Usecases:**
    + Default choice for NLP, transformers, GANs, and most non-vision tasks
    + When you don't have time to extensively tune hyperparameters
    + Research prototyping

## AdamW (Loshchilov & Hutter, 2017)
- **The Update Rule:**  
$\theta_{t+1} = \theta_t - \eta(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t)$  
AdamW performs ***decoupled weight decay***. In standard Adam, L2 regularization is multiplied by the adaptive learning rate $\frac{\eta}{\sqrt{\hat{v}_t}}$, meaning parameters with large gradient variance get less regularization. But regularization should apply equally to all parameters. AdamW moves weight decay outside the adaptive gradient update:
    + ***Adam:*** $\theta \leftarrow \theta - \eta \cdot \frac{g + \lambda \theta}{\sqrt{v}}$
    + ***AdamW:*** $\theta \leftarrow \theta - \eta \cdot \frac{g}{\sqrt{v}} - \eta \lambda \theta$
- **Pros:**
    + Fixes Adam's weight decay behavior
    + Significantly better generalization than Adam
    + The standard optimizer for training Transformers (BERT, GPT, ViT)
- **Usecases:**
    + The default optimizer for modern deep learning, especially transformers and LLMs
    + Replace Adam with AdamW everywhere unless there is a specific reason not to

## Nadam (Nesterov-accelerated Adaptive Moment Estimation)
- **The Update Rule:**  
$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon}(\beta \hat{m}_t + \frac{(1 - \beta_1)g_t}{1 - \beta_1^t})$    
Combines Adam's adaptive learning rates with Nesterov's lookahead momentum. The gradient term includes a blend of the momentum direction and the current gradient in a Nesterov-style formulation.
- **Usecases:**
    + Sometimes marginally better than Adam on certain tasks
    + Not as widely adopted as AdamW

## AMSGrad (Reddi et al., 2018)
- **The Update Rule:**  
$v_t = \beta_2 v_{t-1} + (1 - \beta_2)g_t^2$  
$\hat{v}_t = max(\hat{v}_{t-1}, v_t)\;$  (monotonically increasing second moment)     
$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon}m_t$  
Addresses Adam's convergence issues by ensuring the second moment estimate $\hat{v}_t$ is monotonically non-decreasing. This prevents the adaptive learning rate from increasing over time, which was shown to cause Adam to diverge in some convex counterexamples.
- **Usecases:**
    + Theoretical safety when Adam seems unstable
    + In practice, AdamW is usually preferred; AMSGrad showed limited practical gains

## RAdam (Rectified Adam, Liu et al., 2019)
Adam has high variance in the early stages of training because the second moment estimate is unreliable with few samples. RAdam applies ***a dynamic rectification term that essentially turns off the adaptive learning rate for the first few thousand steps***, behaving like vanilla momentum SGD initially, then smoothly transitioning to full Adam.
- **Usecases:**
    + Drop-in replacement for Adam that often requires less warmup tuning
    + Good for tasks where Adam needs extensive warmup

## LARS / LAMB (Layer-wise Adaptive Rate Scaling)
***LARS (Large Batch Optimization, You et al., 2017)***
***LAMB (Layer-wise Adaptive Moments for Batch training, You et al., 2019)***
- **The Update Rule (LAMB):**
    + Layer-wise update:
     $r_t = \frac{m_t}{\sqrt{v_t} + \epsilon}$  
    + Normalized update:
     $\theta_{t+1}^{(l)} = \theta_t^{(l)} − \eta \cdot \frac{r_t^{(l)}}{∣∣r_t^{(l)}||} \cdot ∣∣\nabla_\theta \mathcal{L}(\theta_t^{(l)})∣∣$   
In very large batch training, standard optimizers fail because the gradient signal becomes too similar across samples, and the effective learning rate behaves strangely. LARS/LAMB normalizes updates per layer and scales them by the layer's weight norm. This allows each layer to update at its own appropriate scale.
- **Pros:**
    + Enables training with batch sizes of 32k–64k without degradation
    + LAMB was used to train BERT in 76 minutes and ALBERT efficiently
    + Critical for distributed training at scale
- **Usecases:**
    + Large-batch distributed training *(TPU pods, large GPU clusters)*
    + When standard Adam/AdamW breaks down at large batch sizes

## Lookahead (Zhang et al., 2019)
- **The Update Rule:**
    + Maintain ***fast weights*** $\phi$  updated by any base optimizer like *(SGD, Adam, etc.)*
    + Every k  steps, update ***slow weights*** $\theta$  toward the average of fast weights:  
    $\theta_{t+1} = \theta_t + \alpha (\phi_t - \theta_t)$ 
    + Reset fast weights to slow weights
***Lookahead*** maintains two sets of parameters. The inner loop explores rapidly with high learning rates, while the outer loop averages these explorations to find a more stable, flat minimum. It's optimizer-agnostic.
- **Pros:**
    + Improves generalization by finding wider minima
    + Reduces variance in the optimization trajectory
    + Minimal overhead; updates slow weights only every k  steps
- **Usecases:**
    + Wrap around Adam or SGD for better generalization
    When training stability is an issue

## SignSGD / Signum (Bernstein et al.)
- **The Update Rule:**  
$\theta_{t+1} = \theta_t − \eta \cdot sign(g_t)$   
    + ***Or with momentum:***
       $m_t = \beta m_{t−1} + (1 − \beta)g_t$  
       $\theta_{t+1} = \theta_t − \eta \cdot sign(m_t)$  
    + ***Extreme compression:*** only 1 bit per gradient coordinate
    + Enables massive distributed training with minimal communication
    + The theoretical foundation for Lion

## Lion (EvoLved Sign Momentum, Chen et al., Google 2023)
- **The Update Rule:**  
$m_t = \beta_1 m_{t−1} + (1 − \beta_1)sign(g_t)$   
$θ_{t+1} = θ_t − \eta \cdot (sign(\beta_1 m_t + (1 − \beta_1)g_t) + \lambda\theta_t)$   
Lion was discovered via evolutionary search on synthetic optimization landscapes. It uses only the sign of the gradient *(like SignSGD)* combined with momentum. This means each parameter update is always $_-^+\eta$  *(plus weight decay)*, making it extremely low-precision friendly.
- **Pros:**
    + Only stores momentum; no second moments like Adam
    + Works well with low-precision *(FP16, BF16, int8)*
    + Outperformed AdamW on ImageNet and language modeling in the original paper
    + Lower communication cost in distributed training; only send signs
- **Cons:**
    + Very sensitive to learning rate and weight decay tuning
    + The sign operation loses gradient magnitude information
    + Still debated whether it consistently beats AdamW across all domains
- **Usecases:**
    + When memory is constrained *(no $v_t$ state needed)*
    + Low-precision training
    + Large-scale distributed training

## Schedule-Free Optimizers (Defazio et al., 2024)
***Optimizers that do not require learning rate schedules***. They maintain two parameter copies — one updated aggressively and one conservatively; and interpolate between them.  
Learning rate scheduling *(warmup, cosine decay, step decay)* is one of the most painful and time-consuming aspects of training deep networks. Schedule-free methods aim to eliminate it entirely while matching or exceeding scheduled SGD/Adam performance.
- **Usecases:**
    + To eliminate learning rate tuning
    + Early-stage research where the right schedule is not known yet

## Muon (Keller Jordan, 2025)
- An optimizer that orthogonalizes the momentum matrix before applying updates. For a gradient matrix G , it computes the update via ***Newton-Schulz iterations*** to find the closest orthogonal matrix to G .
    + Achieves SOTA pretraining efficiency for LLMs, reportedly 2× faster than AdamW in some settings
    + The orthogonalization acts as a powerful preconditioner without explicit second-order information
    + Gained massive attention in the open-source LLM community in 2025
- **Usecases:**
    + LLM pretraining or try for different architectures
    + For cutting-edge training efficiency

## GaLore (Gradient Low-Rank Projection, Zhao et al., 2024)
- Projects gradients into a low-rank subspace before applying the optimizer update. The projection matrices are periodically updated.
    + Reduces memory by up to 65% during LLM training
    + Enables training 7B parameter models on single consumer GPUs
    + Compatible with Adam, AdamW, etc. (it's a wrapper, not a replacement)
- **Usecases:**
    + Memory-constrained LLM training/fine-tuning
    + Full-parameter fine-tuning where LoRA is insufficient

## LOMO / AdaLOMO (Low-Memory Optimization)
Instead of storing full gradients for all parameters, LOMO updates parameters immediately during the backward pass, gradient by gradient, never materializing the full gradient vector.
- Reduces optimizer memory to nearly zero; only parameters needed
- Enables full fine-tuning of 65B+ models on limited hardware
- To fine-tune large models cheaply

# Second Order Methods
Use gradient $\nabla_\theta \mathcal{L}$ and curvature information (Hessian ***H***  or Fisher ***F*** ).
   - ***Computation:*** Multiple gradient evaluations or explicit curvature computation
   - ***Memory:*** O($d^2$)  for exact, O(d)  to O($d^1.5$)  for approximations
   - ***Convergence:*** Quadratic near optimum *(error squares each step)*
   - ***Scale:*** Historically limited to small/medium models; emerging methods challenge this

## Newton's Method
- **The Update Rule:**  
$\theta_{t+1} = \theta_t - \eta \cdot H_t^{-1} \nabla_\theta \mathcal{L}(\theta_t)$   
    Where:  
    $H_t = \nabla^2 \mathcal{L}(\theta_t)$  is the ***Hessian matrix***  
    $H_t^{−1}$  is the inverse Hessian  
    $\eta$ is the step size; often set to 1.0 for pure Newton, or line-searched  
    + Newton's method uses curvature information via the Hessian to take a direct step toward the minimum. If the loss is a quadratic bowl, Newton's method jumps directly to the bottom in one step.  
    + The Hessian defines a local metric on parameter space. The update $−H^{−1}g$  is the steepest descent direction under the metric induced by curvature, not the Euclidean metric.  
- **Pros:**
    + Once close to a minimum, the number of correct digits roughly doubles each iteration making efficient covergence
    + The method is invariant to linear reparameterizations of the problem
    + No need to tune learning rates aggressively
    Handles pathological curvature like long narrow valleys naturally
- **Cons:**
    + Computing the full Hessian is O($d^2$)  in memory and O($d^3$)  in time for inversion, where ***d  = number of parameters***. For a 1B parameter model, this is impossible
    + Not guaranteed to converge from arbitrary starting points; requires positive-definite Hessian
    + In high dimensions, the Hessian has many negative eigenvalues. Pure Newton can actually move toward saddle points or maxima
    + The Hessian of a mini-batch is a terrible estimator of the true Hessian. Exact Newton doesn't work well with stochastic gradients
- **Usecases:**
    + Small-scale convex optimization like ***logistic regression with $d < 10^4$*** 
    + As a theoretical baseline
    + Trust Region Newton methods 

## Natural Gradient / Fisher Information Matrix (Amari, 1998)
- **The Update Rule:**  
$\theta_{t+1} = θ_t − \eta \cdot F_t^{−1} \nabla_\theta \mathcal{L}(\theta_t)$   
Where $F_t$  is the ***Fisher Information Matrix***  
Standard gradient descent moves in Euclidean parameter space. But parameters that are *close* in Euclidean distance may define probability distributions that are very different. Natural gradient moves in the space of probability distributions using the Fisher metric, which measures distributional distance via ***KL divergence***.    
The Fisher matrix is ***the expected Hessian of the negative log-likelihood, or equivalently the covariance of the score function***. Under certain conditions, it approximates the Hessian but is always positive semi-definite.
    + $F^{−1}g$  is the direction of steepest descent in KL divergence
    + Equivalent to Newton's method when the loss is the negative log-likelihood and the model is in the exponential family
    + Invariant to reparameterization of the probability distribution
- **Pros:**
    + Theoretically elegant; moves in the ***right geometry for probabilistic models***
    + Often more stable than the raw Hessian 
    + Better conditioning than Euclidean gradient descent
- **Cons:**
    + Computing and inverting the full Fisher is just as expensive as the Hessian: O($d^2$)  memory, O($d^3$)  time
    + For deep networks with millions/billions of parameters, exact natural gradient is intractable
- **Usecases:**
    + Small probabilistic models
    + The theoretical foundation for all approximate second-order methods in deep learning 8(K-FAC, etc.)*

## L-BFGS (Limited-memory Broyden-Fletcher-Goldfarb-Shanno)
Instead of storing the full ***d×d  Hessian (or its inverse)***, L-BFGS stores only the last m  gradient vectors and parameter differences *(typically m=10  to 100 )*. It uses these to implicitly construct a low-rank approximation of the inverse Hessian via the secant equation.
- ***The Update Rule (two-loop recursion)***
Using history {$s_k = θ_{k+1} − θ_k$}  and {$y_k = \nabla_{k+1} − \nabla_k$} , L-BFGS computes $H^{−1}g$  without ever forming H :

    + Initialize q = $\nabla_t$ 
    + For i = t−1,…,t−m : $\alpha_i$ = $\rho_i s_i^T q$ ; q = q − $α_i y_i$ 
    + Scale: $r = H_0q$  (where $H_0$ is a diagonal approximation)
    + For i = t−m,…,t−1 : $\beta = \rho_i y_i^T r$; r = r + $s_i(\alpha_i - \beta)$ 
    + Update: $\theta_{t+1} = \theta_t − \eta \cdot r$ 
    + Where $\rho_i = \frac{1}{y_i^T s_i}$
- **Pros:**
    + Memory: O(md)  instead of O($d^2$); manageable even for millions of parameters
    + No explicit Hessian computation needed
    + Superlinear convergence; slower than Newton's quadratic, but much faster than first-order linear
    + Very effective for deterministic, full-batch optimization
- **Cons:**
    + Requires line searches to guarantee positive-definite approximations
    + Breaks down with mini-batches: The secant equation assumes consistent gradients. With stochastic gradients, $y_k$  is noisy and the implicit Hessian approximation becomes garbage. L-BFGS is essentially unusable with small batches in deep learning.
    + No natural way to incorporate momentum or adaptive learning rates
    + Struggles with non-convex landscapes like saddle points, cliffs
- **Usecases:**
    + Full-batch optimization of small-to-medium models
    + Traditional ML *(logistic regression, SVMs, CRFs)* where you can compute exact gradients
    + As a component in hybrid methods or meta-optimization
- **Variants:**
    + ***oL-BFGS / Stochastic L-BFGS:*** Attempts to make L-BFGS work with mini-batches by adding regularization or damping. Limited success in deep learning.
    + ***Online L-BFGS:*** For online convex optimization. Not widely used in modern deep learning.

## Hessian-Free Optimization (HF / Newton-CG, Martens 2010)
Instead of computing $H^{−1}$  explicitly, solve the linear system $H \cdot p = −\nabla \mathcal{L}$  for the update direction p  using the ***Conjugate Gradient algorithm***.  
CG only needs Hessian-vector products $H \cdot v$ , not the full Hessian matrix. These can be computed efficiently via Pearlmutter's trick of automatic differentiation of gradient-vector products in O(d)  time and memory.
- **The Algorithm:**
    + Compute gradient $\nabla \mathcal{L}$
    + Use CG to solve $H \cdot p = −\nabla \mathcal{L}$ for p 
        * Each CG iteration requires one Hessian-vector product
        * Run CG for k$\ll$d  iterations for early truncation
    + Update $\theta \leftarrow \theta + \eta \cdot p$
    + Use damping *(Levenberg-Marquardt style)*: solve $(H + \lambda I) \cdot p = −\nabla \mathcal{L}$ to ensure positive-definiteness
- **Pros:**
    + Exact second-order information without storing the Hessian
    + Works with mini-batches; though damping and CG iterations need tuning
    + Can use Gauss-Newton Hessian instead of full Hessian for greater stability
    + Once state-of-the-art for deep autoencoders and RNNs 
- **Cons:**
    + Each update requires k  forward/backward passes *(for CG iterations)* which is expensive
    + CG convergence is slow if the Hessian is ill-conditioned
    + Very sensitive to damping parameter $\lambda$ 
    + Largely superseded by K-FAC and adaptive methods for large-scale training
- **Usecases:**
    + Medium-scale problems where curvature matters *(deep autoencoders, physics-informed networks)*
    + When you need true second-order information but can't store the full Hessian
    + Historical importance for understanding the trajectory toward K-FAC

## K-FAC (Kronecker-Factored Approximate Curvature, Martens & Grosse, 2015)
The Fisher Information Matrix for a deep neural network has special structure. For a layer with ***weight matrix W*** *(or fully-connected layer)*, the Fisher can be approximated as a ***Kronecker product*** of two much smaller matrices:  
$F \approx A \otimes G$   
Where:  
    A = $\mathbb{E}[aa^T]$  is the covariance of activations; inputs to the layer  
    G = $\mathbb{E}[gg^T]$  is the covariance of output gradients; backpropagated derivatives  
For a linear layer $y = W x$ , the derivative with respect to $W$  is the outer product of the backpropagated gradient and the input activation. The Fisher *(expected outer product of gradients)* naturally factorizes.
- **The Update Rule:**  
Instead of inverting the full $d \times d$  Fisher, invert the Kronecker factors:
$F^{−1} \approx A^{−1} \otimes G^{−1}$  
For a layer with m  inputs and n  outputs:  
    A  is m$\times$m  
    G  is n$\times$n  
    Inversion cost: O($m^3+n^3$)  instead of O($(mn)^3$) 
- **Pros:**
    + Captures layer-wise curvature much better than diagonal approximations
    + More tractable than full Fisher or Hessian
    + Empirically converges faster than SGD on some deep networks
    + Can be made more efficient with eigen-decomposition caching
- **Cons:**
    + Assumes Kronecker structure which is only exact for fully-connected layers. Convolutions, attention, and normalization layers require approximations.
    + Still expensive compared to first-order methods as it requires storing and inverting A and G per layer
    + The A  and G  estimates from mini-batches can be noisy
    + Complex implementation; not natively supported in most frameworks
- **Usecases:**
    + Medium-to-large fully-connected networks
    + When first-order methods plateau and curvature is the bottleneck
    + Research into optimization for deep learning
- **Variants:**
    + ***K-FAC with momentum and damping:*** Essential for stability
    + ***EK-FAC (Eigenvalue-corrected K-FAC):*** Improves the Kronecker approximation by tracking eigenvalue corrections
    + ***UK-FAC / Trace estimation:*** Further approximations for convolutions

## Sophia (Second-order Clipped Optimizer, Liu et al., 2023)
Sophia is designed to bring second-order benefits to LLM pretraining without the prohibitive cost of full Hessian methods. It uses a diagonal Hessian approximation estimated via the ***Hutchinson estimator*** *(randomized finite differences)* and clips updates by curvature.
- **The Update Rule:**  
$h_t$ = Clip(HutchinsonEstimate($theta_t$), $\rho$)  
$\theta_{t+1} = \theta_t - \eta \cdot \frac{m_t}{max(h_t, \epsilon)}$  
Where:
    + $m_t$  is the EMA*(Exponential Moving Average)* of gradients *(momentum)*  
    + $h_t$ is the estimated diagonal Hessian  
    + Clip  ensures the Hessian estimate doesn't explode  
    + The update is essentially gradient divided by diagonal Hessian *(like AdaGrad/Adam but with true curvature instead of squared gradients)*   
***Hutchinson Estimator***  
$v^THv = lim_{\delta \rightarrow 0} \frac{\nabla \mathcal{L}(\theta + \delta v) − \nabla \mathcal{L}(\theta - \delta v)}{2 \delta}$   
For random $v \sim \mathcal{N}(0, I)$,  
$\mathbb{E}[v \otimes (Hv)] = diag(H)$  
- **Pros:**
    + 2× faster than AdamW for LLM pretraining (GPT-2 scale) in the original paper
    + Uses actual curvature *(Hessian diagonal)* rather than squared-gradient heuristics
    + The clipping makes it stable despite stochastic Hessian estimates
    + Memory overhead is moderate
- **Cons:**
    + Requires two extra gradient evaluations per Hessian update step; for Hutchinson estimation
    + The diagonal approximation ignores off-diagonal curvature; the interactions between parameters
    + Still relatively new; not yet universally adopted over AdamW
    + Theoretical guarantees are weaker than full second-order methods
- **Usecases:**
    + LLM pretraining and faster convergence than AdamW
    + The extra gradient evaluations with computational power 
    + As a bridge between first-order Adam and full second-order methods

## Shampoo (Gupta et al., 2018) & Distributed Shampoo (Anil et al., 2020)
Shampoo is a full-matrix preconditioning method that ***generalizes AdaGrad***. Instead of maintaining a diagonal matrix of squared gradients, it maintains full matrices for each dimension of a tensor parameter.  
- For a matrix parameter $W \in R^{m \times n}$ :  
    + Maintain $L_t \in R^{m \times m}$  (accumulating gradient outer products along columns)
    + Maintain $R_t \in R^{n \times n}$  (accumulating gradient outer products along rows)
- **The Update Rule:**  
$L_{t+1} = L_t + G_t G_t^T\; ,\; R_{t+1} = R_t + G_t^T G_t$   
$W_{t+1} = W_t − \eta \cdot L_t^{−1/4} G_t R_t^{−1/4}$   
The full preconditioner would be (L$\otimes$R)$^{−1/2}$ , but since we apply it to the matrix gradient directly, we use matrix fourth-roots to get the equivalent effect.  
- **Pros:**
    + Captures full matrix structure within each tensor dimension
    + More expressive than diagonal methods but cheaper than full Hessian
    + Works well for matrix and tensor parameters common in deep learning
- **Cons:**
    + Computing matrix inverse fourth roots is expensive as it requires SVD or eigendecomposition. Distributed Shampoo addresses this by computing roots periodically and caching.
- **Usecases:**
    + Large-scale training where AdamW plateaus
    + Google's internal workloads; increasingly available in frameworks

## SOAP (Shampoo with Orthogonalization Preconditioning, Vyas et al., 2025)
A recent improvement that combines **Shampoo's preconditioning with orthogonalization; inspired by Muon**. SOAP applies Shampoo's gradient transformation but then orthogonalizes the update matrix.
- **Pros:**
    + Outperforms both Shampoo and AdamW in recent benchmarks
    + Better conditioning than Shampoo alone
    + Represents the current frontier of practical second-order-ish methods

## Gauss-Newton Method & Generalized Gauss-Newton (GGN)
For a loss $\; \mathcal{L} = \sum_i \ell(f(x_i ; \theta) , y_i)$ , the Hessian is:  
$H = \sum_i J_i^T H_{\ell, i} J_i $ + terms involving $\nabla_\theta f$   
The Gauss-Newton approximation drops the second-order terms involving $\nabla_\theta f$ :  
$H_{GN} = \sum_i J_i^T H_{\ell, i} J_i \approx \sum_i J_i^T J_i \; \; (when \; H_{\ell, i} \approx  I)$   
Where $J_i = \frac{\delta f (x_i)}{\delta \theta}$  is the Jacobian of the network output.
   - $H_{GN}$  is guaranteed positive semi-definite; unlike the full Hessian
   - Natural fit for least-squares and maximum likelihood
   - Used implicitly in K-FAC; which approximates the Fisher, itself a Gauss-Newton approximation
   - Foundation for many second-order methods in neural networks

## Trust Region Methods / TRPO (Schulman et al., 2015)
Instead of a fixed learning rate, constrain the update to a region where the quadratic approximation is valid:  
$min_p \nabla^T p + \frac{1}{2} p^T Hp  \; \; \text{s.t.} ∣∣p∣∣\le \nabla$  
- **TRPO in RL:**
Uses the Fisher Information Matrix *(natural gradient)* with a KL-divergence trust region:  
$max_\theta \mathbb{E}[\frac{\pi_\theta}{\pi_\theta(old)} A] \; \; \text{s.t.} D_{KL} (\pi_\theta(old)∣∣\pi_\theta) \le \delta$  
- **Usecases:**
    + Reinforcement learning policy optimization *(TRPO, PPO uses a clipped surrogate instead)*
    + When step size stability is more important than raw speed

# Comparative Discussion Between 1st and 2nd Order Methods:

| Dimension                          | First-Order (SGD, Adam, etc.)                                            | Second-Order (Newton, K-FAC, etc.)                                                                                               |
| ---------------------------------- | ------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------------------------- |
| **Information Used**               | Gradient vector $\nabla \mathcal{L} \in \mathbb{R}^d$                    | Hessian/Fisher $H \in \mathbb{R}^{d \times d}$ or approximations                                                                 |
| **Convergence Rate**               | **Linear**: error $\propto (1 - \frac{1}{\kappa})^t$                     | **Quadratic**: error $\propto \varepsilon_{t+1} \propto \varepsilon_t^2$ (exact Newton near optimum)                             |
| **Per-Iteration Cost**             | $O(d)$ memory, $O(d)$ time (one forward-backward)                        | Exact: $O(d^2)$ memory, $O(d^3)$ time. Approximate: $O(d)$ to $O(d^{1.5})$                                                       |
| **Memory Footprint**               | 1×–3× parameters (SGD: 1×, Adam: 2×, Lion: 1×)                           | Exact: $d^2$ entries. L-BFGS: $2md$. K-FAC: layer-wise matrices. Sophia: +1× for diagonal Hessian                                |
| **Stochastic Behavior**            | Naturally handles mini-batch noise                                       | Struggles; Hessian of a mini-batch is a poor estimator. Requires damping, large batches, or variance reduction                   |
| **Saddle Point Handling**          | Can escape slowly via noise/momentum                                     | Newton moves *toward* saddle points (attracted by negative curvature). Must use damping or trust regions                         |
| **Hyperparameter Sensitivity**     | Highly sensitive to learning rate, schedules                             | Less sensitive to step size (affine invariant), but sensitive to damping, trust region radius, batch size                        |
| **Scale Invariance**               | Not invariant. Learning rate must be tuned per problem, per architecture | Newton is affine invariant. Natural gradient is reparameterization invariant                                                     |
| **Non-Convex Landscapes**          | Robust. Noise and mini-batch variance act as implicit regularization     | Fragile. Curvature estimates can be misleading far from minima. Approximations break down                                        |
| **Practical Scale**                | Billions of parameters (LLMs, vision transformers)                       | Millions of parameters (exact). Tens of millions (K-FAC). Billions only with extreme approximations (diagonal Hessian in Sophia) |
| **Wall-Clock Time to Convergence** | Many iterations, but each is fast. Often wins in wall-clock time for DL  | Fewer iterations, but each is much slower. Rarely wins in wall-clock time except in specialized settings                         |


## Why Second-Order Methods Struggle in Modern Deep Learning:
1. **The Stochasticity Problem**
Second-order methods assume you can estimate curvature reliably. But with a mini-batch of 512 samples from a dataset of $10^9$, the Hessian or Fisher estimate is incredibly noisy. First-order methods are robust to this noise because the expected gradient is correct; the expected Hessian is also correct in theory, but the variance is catastrophic, and inversion amplifies noise.
2. **The Saddle Point Epidemic**
In high dimensions ($d \sim 10^9$), local minima are rare; saddle points are exponentially more common. The Hessian at a saddle point has both positive and negative eigenvalues. Pure Newton's method ($−H^{−1}g$ ) will move toward the saddle along negative curvature directions unless you explicitly modify the Hessian (damping, trust regions, or using the absolute value of eigenvalues).
3. **The Memory Wall**
A 1B parameter model has d=109 . The Hessian requires 1018  entries (8 exabytes in float32). Even storing a modest approximation is prohibitive. This is why all practical second-order methods in deep learning use:
    - Diagonal approximations (Sophia, AdaGrad)
    - Low-rank approximations (L-BFGS)
    - Kronecker approximations (K-FAC, Shampoo)
    - Implicit methods (Hessian-Free CG)
4. **The Generalization Paradox**
Paradoxically, first-order methods often generalize better than second-order methods in deep learning.  
    - Second-order methods converge to sharp minima (high curvature) more easily
    - First-order methods with noise and small learning rates prefer flat minima that generalize better
    - The "implicit regularization" of first-order noise may be beneficial
5. Adam's $v_t$  *(exponential moving average of squared gradients)* is a very crude diagonal approximation of the Fisher Information Matrix. It estimates curvature by assuming $H \approx diag(g^2)$ . Methods like Sophia replace this heuristic with an actual Hessian diagonal estimate. Methods like K-FAC and Shampoo go further by capturing off-diagonal structure within layers.

| Scenario                                  | Recommended Approach                                                    | Rationale                                                       |
| ----------------------------------------- | ----------------------------------------------------------------------- | --------------------------------------------------------------- |
| **LLM pretraining (1B+ params)**          | AdamW (default), Sophia (experimental), Muon (emerging)                 | Memory constraints dominate; first-order + good scheduling wins |
| **Vision model training (ResNet, ViT)**   | SGD + Momentum + cosine decay, or AdamW                                 | Generalization is key; first-order methods find flatter minima  |
| **Small model (< 1M params), full batch** | L-BFGS or Newton                                                        | Exact second-order is tractable and converges much faster       |
| **Medium model, curvature bottleneck**    | K-FAC, Shampoo, or Hessian-Free                                         | Captures some second-order structure without full Hessian       |
| **RL policy optimization**                | TRPO (natural gradient + trust region), PPO (first-order approximation) | Step size constraints prevent policy collapse                   |
| **Meta-learning / hyperparameter opt**    | Second-order or approximate second-order                                | You need curvature through the inner optimization               |
| **Ill-conditioned convex problem**        | Newton or L-BFGS                                                        | Guaranteed convergence; no saddle points                        |
| **Online / non-stationary learning**      | AdaGrad, Adam (first-order adaptive)                                    | Second-order estimates become stale quickly                     |
